# 6.1

Подключим необходимые библиотеки

In [14]:
import numpy as np
import random

Введем функцию для деления многолченов с остатком

In [15]:
def polynomial_division_remainder(dividend, divisor):
    # Создаем копию делимого для изменения в процессе деления
    remainder = list(dividend)
    
    # Длина делимого и делителя
    len_divisor = len(divisor)
    
    # Пока степень остатка >= степени делителя
    while len(remainder) >= len_divisor:
        # Находим сдвиг для делителя, чтобы выровнять его с остатком
        shift = len(remainder) - len_divisor
        
        # Выполняем XOR между делителем, сдвинутым на shift позиций, и остатком
        for i in range(len_divisor):
            remainder[shift + i] ^= divisor[i]
        
        # Удаляем все последние нули из остатка, чтобы уменьшить его степень
        while len(remainder) > 0 and remainder[len(remainder) - 1] == 0:
            remainder = remainder[:len(remainder) - 1]

    return np.array(remainder)

Введем функцию для умножения многочленов

In [16]:
def polynomial_multiply(A, B):
    degree_A = len(A)
    degree_B = len(B)
    result = np.zeros(degree_A + degree_B - 1, dtype=int)  # явное указание типа int
    
    for i in range(degree_B):
        if B[i] == 1:  # если коэффициент в B ненулевой
            result[i:i + degree_A] ^= A.astype(int)  # конвертируем A в целочисленный тип, если нужно

    return result


Введем функцию для допущения n-кратной ошибки в сообщении и попытки ее исправления

In [17]:
def make_and_correct_error(a, g, error_rate):
    print("Входное сообщение:      ", a)
    print("Порождающий полином:    ", g)

    v = polynomial_multiply(a, g)
    print("Отправленное сообщение: ", v)

    w = v.copy()
    error = np.zeros(len(w), dtype=int)

    if error_rate == 1:
        # Однократная ошибка — случайный индекс
        index = random.randint(0, len(w) - 1)
        error[index] = 1
    elif error_rate == 2:
        # Двухкратная ошибка в пределах 3 соседних разрядов
        index1 = random.randint(0, len(w) - 2)
        index2 = index1 + random.choice([1, 2])
        error[index1] = 1
        error[index2] = 1
    else:
        # Для всех остальных случаев (больше двух ошибок) ошибки ставятся случайно
        error_indices = random.sample(range(w.shape[0]), error_rate)
        for index in error_indices:
            error[index] = 1

    w = (w + error) % 2
    print("Сообщение с ошибкой:    ", w)

    s = polynomial_division_remainder(w, g)
    error_templates = None
    if error_rate == 1:
        error_templates = [[1]]
    else:
        error_templates = [[1, 1, 1], [1, 0, 1], [1, 1], [1]]

    idx = 0
    found = False
    for template in error_templates:
        if np.array_equal(s, template):
            found = True
    while not found:
        s = polynomial_division_remainder(polynomial_multiply(s, np.array([0, 1])), g)
        for template in error_templates:
            if np.array_equal(s, template):
                found = True
        idx += 1

    temp = np.zeros(len(w), dtype=int)
    if idx == 0:
        temp[idx] = 1
    else:
        temp[len(temp) - idx] = 1

    e = polynomial_multiply(s, temp)
    e = e[:len(w)]
    message = (w + e) % 2
    print("Исправленное сообщение: ", message)
    
    if np.array_equal(v, message):
        print("Ошибка исправлена корректно")
    else:
        print("Ошибка исправлена некорректно")


Введем входное сообщение a и порождающий полином g = 1 + x^2 + x^3

In [18]:
a = np.array([1, 0, 0 ,1])
g = np.array([1, 0, 1, 1])

Проведем исследование для однократной ошибки

In [19]:
make_and_correct_error(a, g, 1)

Входное сообщение:       [1 0 0 1]
Порождающий полином:     [1 0 1 1]
Отправленное сообщение:  [1 0 1 0 0 1 1]
Сообщение с ошибкой:     [1 0 1 0 0 0 1]
Исправленное сообщение:  [1 0 1 0 0 1 1]
Ошибка исправлена корректно


Проведем исследование для двухкратной ошибки

In [20]:
make_and_correct_error(a, g, 2)

Входное сообщение:       [1 0 0 1]
Порождающий полином:     [1 0 1 1]
Отправленное сообщение:  [1 0 1 0 0 1 1]
Сообщение с ошибкой:     [1 1 1 1 0 1 1]
Исправленное сообщение:  [0 0 0 1 0 1 1]
Ошибка исправлена некорректно


Проведем исследование для трехкратной ошибки

In [21]:
make_and_correct_error(a, g, 3)

Входное сообщение:       [1 0 0 1]
Порождающий полином:     [1 0 1 1]
Отправленное сообщение:  [1 0 1 0 0 1 1]
Сообщение с ошибкой:     [0 1 1 1 0 1 1]
Исправленное сообщение:  [0 1 1 1 0 1 0]
Ошибка исправлена некорректно


Код (7, 4) предназначен для исправления лишь однократных ошибок, поэтому в случае двухкратных и трехкратных ошибок сообщение исправляется некорректно

# 6.2

Введем новые входное сообщение a и порождающий полином g = 1 + x^3 + x^4 + x^5 + x^6

In [22]:
a = np.array([1, 0, 0, 1, 0, 0, 0, 1, 1])
g = np.array([1, 0, 0, 1, 1, 1, 1])

Проведем исследование для однократной ошибки

In [23]:
make_and_correct_error(a, g, 1)

Входное сообщение:       [1 0 0 1 0 0 0 1 1]
Порождающий полином:     [1 0 0 1 1 1 1]
Отправленное сообщение:  [1 0 0 0 1 1 0 0 0 1 1 0 0 0 1]
Сообщение с ошибкой:     [1 0 0 1 1 1 0 0 0 1 1 0 0 0 1]
Исправленное сообщение:  [1 0 0 0 1 1 0 0 0 1 1 0 0 0 1]
Ошибка исправлена корректно


Проведем исследование для двухкратной ошибки

In [24]:
make_and_correct_error(a, g, 2)

Входное сообщение:       [1 0 0 1 0 0 0 1 1]
Порождающий полином:     [1 0 0 1 1 1 1]
Отправленное сообщение:  [1 0 0 0 1 1 0 0 0 1 1 0 0 0 1]
Сообщение с ошибкой:     [1 0 0 0 1 1 0 0 0 1 1 0 0 1 0]
Исправленное сообщение:  [1 0 0 0 1 1 0 0 0 1 1 0 0 0 1]
Ошибка исправлена корректно


Проведем исследование для трехкратной ошибки

In [25]:
make_and_correct_error(a, g, 3)

Входное сообщение:       [1 0 0 1 0 0 0 1 1]
Порождающий полином:     [1 0 0 1 1 1 1]
Отправленное сообщение:  [1 0 0 0 1 1 0 0 0 1 1 0 0 0 1]
Сообщение с ошибкой:     [1 0 0 0 1 1 0 0 0 0 0 0 0 0 0]
Исправленное сообщение:  [1 0 0 0 1 1 0 1 1 1 0 0 0 0 0]
Ошибка исправлена некорректно


Проведем исследование для четырехкратной ошибки

In [26]:
make_and_correct_error(a, g, 4)

Входное сообщение:       [1 0 0 1 0 0 0 1 1]
Порождающий полином:     [1 0 0 1 1 1 1]
Отправленное сообщение:  [1 0 0 0 1 1 0 0 0 1 1 0 0 0 1]
Сообщение с ошибкой:     [1 0 1 0 0 0 0 0 0 1 1 0 0 1 1]
Исправленное сообщение:  [1 0 1 0 0 0 0 0 0 1 1 1 1 0 1]
Ошибка исправлена некорректно


Код (15, 9) предназначен для исправления однократных и двухкратных ошибок, поэтому трехкратные и четырехкратные ошибки исправляются некорректно